# 03 — Pneumonia Detection: VGG16 Transfer Learning

Two-phase training of a VGG16-based classifier on the chest X-ray dataset:

| Phase | Strategy | Learning Rate |
|-------|----------|---------------|
| **Phase 1** | Freeze all VGG16 conv layers; train dense head only | 1e−4 |
| **Phase 2** | Unfreeze block5 (last conv block); fine-tune end-to-end | 1e−5 |

Both phases use class weights (`{0: 1.0, 1: 4.0}`), in-graph augmentation, and VGG16 preprocessing applied internally by the model.

**Sections**
1. Environment Setup
2. Configuration
3. Data Loading & Preprocessing
4. VGG16 Training
5. Evaluation
6. Conclusion — Transfer Learning Model Comparison

## 1. Environment Setup

Import libraries and set random seeds for reproducibility.

In [ ]:
import os
import sys
import json
import random
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras
from keras import layers

from helpers import data_utils, model_utils, training_utils, visualization

keras.utils.set_random_seed(42)
np.random.seed(42)
random.seed(42)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)


## 2. Configuration

Dataset paths, image size, checkpoint locations, and runtime flags.

In [ ]:
USE_COLAB = True
USE_KAGGLE = False  # Set True when running on Kaggle

In [ ]:
# Configuration
if USE_KAGGLE:
    DATASET_ROOT = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/"
    SAVE_DIR = pathlib.Path("/kaggle/working/saved_models")
elif USE_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_ROOT = "/content/drive/MyDrive/x-ray-dataset/"
    SAVE_DIR = pathlib.Path("/content/drive/MyDrive/saved_models")
else:
    DATASET_ROOT = "dataset"
    SAVE_DIR = pathlib.Path("saved_models")

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR   = os.path.join(DATASET_ROOT, "val")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT  = 0.1
AUTOTUNE   = tf.data.AUTOTUNE

for split_path in [TRAIN_DIR, TEST_DIR]:
    if not os.path.isdir(split_path):
        raise FileNotFoundError(f"Missing directory: {split_path}")

SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Model checkpoint paths
TEACHER_CHECKPOINT_PATH  = SAVE_DIR / "vgg16_frozen_best_checkpoint.keras"
FINETUNE_CHECKPOINT_PATH = SAVE_DIR / "vgg16_finetuned_best_checkpoint.keras"

runtime_name = "Kaggle" if USE_KAGGLE else ("Google Colab" if USE_COLAB else "Local")
print(f"Environment:  {runtime_name}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Save dir:     {SAVE_DIR}")


In [ ]:
# Model saver bound to SAVE_DIR
save_model_with_meta = training_utils.make_model_saver(SAVE_DIR)

## 3. Data Loading & Preprocessing

Raw [0, 255] images are loaded — VGG16 preprocessing (`vgg16.preprocess_input`) is applied **inside the model** after augmentation.  This avoids double-preprocessing and ensures augmentation operates on natural pixel values.

In [ ]:
# Gather train + val + test paths and labels
all_train_paths, all_train_labels, test_paths, test_labels = data_utils.gather_split_paths_labels(
    TRAIN_DIR,
    TEST_DIR,
    val_dir=VAL_DIR,
)

if len(np.unique(all_train_labels)) < 2:
    raise ValueError("Both NORMAL and PNEUMONIA classes must exist in the training pool.")

print(f"Training pool size: {len(all_train_paths)} images")
print(f"  NORMAL={(all_train_labels==0).sum()}, PNEUMONIA={(all_train_labels==1).sum()}")
print(f"Test set size: {len(test_paths)} images")

### 3.1 Augmentation & Class Weights

Define the augmentation pipeline and class weights. The weight of 4.0 for PNEUMONIA penalises missed positive cases, reflecting the clinical priority of minimising false negatives.

In [ ]:
data_augmentation = data_utils.build_data_augmentation(
    rotation=30,
    width_shift=0.1,
    height_shift=0.1,
    shear=0.2,
    zoom=0.2,
)

# Raw [0,255] images — no preprocess_fn (model handles VGG16 preprocessing)
train_ds, val_ds, test_ds, ds_meta = data_utils.build_train_val_test_datasets(
    all_train_paths,
    all_train_labels,
    test_paths,
    test_labels,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    seed=42,
    autotune=AUTOTUNE,
)

# Penalise missed pneumonia 4x (clinical priority: minimise false negatives)
class_weights = {0: 1.0, 1: 4.0}

print(f"Train split -> NORMAL={ds_meta['normal_count']}, PNEUMONIA={ds_meta['pneumonia_count']}")
print(f"Validation size: {len(ds_meta['val_labels'])}")
print("Class weights:", class_weights)
print("Datasets are ready.")

## 4. VGG16 Training

### 4.1 Phase 1 — Frozen Feature Extraction

All VGG16 convolutional layers are frozen.  Only the newly added dense head (GlobalAveragePooling2D → Dense 256 → BN → Dropout → Dense 128 → BN → Dropout → Dense 1) is trained.  Learning rate: **1e−4**, patience: **7**, monitored metric: **val_auc**.

In [ ]:
teacher_model = model_utils.build_vgg16_model(
    img_size=IMG_SIZE,
    augmentation_layer=data_augmentation,
    dense_units=256,
    dense_units_2=128,
    dropout=0.5,
    dropout_2=0.3,
    learning_rate=1e-4,
    freeze_base=True,
    name="vgg16_frozen",
)
teacher_model.summary(show_trainable=True)

teacher_callbacks = training_utils.get_training_callbacks(
    checkpoint_path=TEACHER_CHECKPOINT_PATH,
    patience=7,
    monitor="val_auc",
)

teacher_history = teacher_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=teacher_callbacks,
    class_weight=class_weights,
    verbose=1,
)
print(f"Phase 1 complete. Checkpoint: {TEACHER_CHECKPOINT_PATH}")

### 4.2 Phase 2 — Fine-Tuning Block5

The best Phase 1 checkpoint is loaded and the last convolutional block (`block5_conv1`, `block5_conv2`, `block5_conv3`, `block5_pool`) is unfrozen.  A 10× smaller learning rate (**1e−5**) is used to avoid catastrophic forgetting of the lower-level features learned on ImageNet.  Patience is reduced to **5** epochs.

In [ ]:
# Phase 1 training curves
visualization.plot_combined_training_curves(
    teacher_history,
    phase1_label="Phase 1 – Frozen Base",
    phase2_label="(fine-tune not started yet)",
)

# Load Phase 1 checkpoint and unfreeze block5
frozen_model   = training_utils.load_model_compat(TEACHER_CHECKPOINT_PATH)
finetune_model = model_utils.unfreeze_vgg16_top_block(frozen_model, learning_rate=1e-5)

finetune_callbacks = training_utils.get_training_callbacks(
    checkpoint_path=FINETUNE_CHECKPOINT_PATH,
    patience=5,
    monitor="val_auc",
)

finetune_history = finetune_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=finetune_callbacks,
    class_weight=class_weights,
    verbose=1,
)
print(f"Phase 2 complete. Checkpoint: {FINETUNE_CHECKPOINT_PATH}")

## 5. Evaluation

### 5.1 Combined Training Curves & Threshold Tuning

Plot stitched Phase 1 + Phase 2 training history and tune decision thresholds on the validation set (maximising PNEUMONIA F1).

In [ ]:
# Training curves + threshold tuning
visualization.plot_combined_training_curves(
    teacher_history,
    finetune_history,
    phase1_label="Phase 1 - Frozen Base",
    phase2_label="Phase 2 - Fine-tune (block5)",
)

# Load best checkpoints
frozen_best   = training_utils.load_model_compat(TEACHER_CHECKPOINT_PATH)
finetune_best = training_utils.load_model_compat(FINETUNE_CHECKPOINT_PATH)

# Tune threshold on validation set (maximise PNEUMONIA F1)
frozen_threshold,   frozen_val_f1,   _, _ = training_utils.tune_threshold(frozen_best,   val_ds)
finetune_threshold, finetune_val_f1, _, _ = training_utils.tune_threshold(finetune_best, val_ds)

print(f"VGG16 Frozen    — val F1={frozen_val_f1:.3f}  @ threshold={frozen_threshold:.2f}")
print(f"VGG16 Finetuned — val F1={finetune_val_f1:.3f}  @ threshold={finetune_threshold:.2f}")

### 5.2 Test Set Metrics

Evaluate both models on the held-out test set and save metadata.

In [ ]:
# Test set metrics for both models
frozen_metrics,   frozen_report,   y_true, frozen_prob,   frozen_pred   = training_utils.evaluate_model(frozen_best,   test_ds, frozen_threshold)
finetune_metrics, finetune_report, y_true, finetune_prob, finetune_pred = training_utils.evaluate_model(finetune_best, test_ds, finetune_threshold)

print("── VGG16 Frozen (Phase 1) ──────────────────────────────────")
print(f"  Accuracy:    {frozen_metrics['accuracy']:.4f}")
print(f"  AUC:         {frozen_metrics['auc']:.4f}")
print(f"  Precision:   {frozen_metrics['precision']:.4f}")
print(f"  Recall:      {frozen_metrics['recall']:.4f}")
print(f"  F1:          {frozen_metrics['f1']:.4f}")
print(f"  Threshold:   {frozen_threshold:.2f}")

print("\n── VGG16 Fine-tuned (Phase 2) ──────────────────────────────")
print(f"  Accuracy:    {finetune_metrics['accuracy']:.4f}")
print(f"  AUC:         {finetune_metrics['auc']:.4f}")
print(f"  Precision:   {finetune_metrics['precision']:.4f}")
print(f"  Recall:      {finetune_metrics['recall']:.4f}")
print(f"  F1:          {finetune_metrics['f1']:.4f}")
print(f"  Threshold:   {finetune_threshold:.2f}")

# Save both models with metadata
save_model_with_meta(frozen_best,   "vgg16_frozen",    frozen_metrics,   teacher_history,  {"lr": 1e-4, "freeze_base": True},                    frozen_threshold)
save_model_with_meta(finetune_best, "vgg16_finetuned", finetune_metrics, finetune_history, {"lr": 1e-5, "freeze_base": False, "unfrozen": "block5"}, finetune_threshold)

### 5.3 Confusion Matrices

Side-by-side confusion matrices with counts and row-normalised percentages.

In [ ]:
visualization.plot_confusion_matrices_grid([
    ("VGG16 Frozen",     y_true, frozen_pred),
    ("VGG16 Fine-tuned", y_true, finetune_pred),
])

### 5.4 ROC Curves

Overlaid ROC curves with AUC in the legend and the operating point (tuned threshold) marked.

In [ ]:
visualization.plot_roc_curves([
    ("VGG16 Frozen",     y_true, frozen_prob,   frozen_threshold),
    ("VGG16 Fine-tuned", y_true, finetune_prob, finetune_threshold),
])

### 5.5 Sample Predictions

Qualitative predictions from the fine-tuned model on test images.

In [ ]:
# Sample predictions from fine-tuned model
visualization.plot_sample_predictions(finetune_best, test_ds, finetune_threshold, n=9)

## 6. Conclusion — Transfer Learning Model Comparison

Compare both VGG16 variants and all previous CNN baselines.

In [ ]:
records = []
for model_name, label in [
    ("baseline_cnn",       "Baseline CNN"),
    ("baseline_cnn_tuned", "Baseline CNN (tuned HP)"),
    ("vgg16_frozen",       "VGG16 Frozen (Phase 1)"),
    ("vgg16_finetuned",    "VGG16 Fine-tuned (Phase 2)"),
]:
    meta = training_utils.load_model_meta(SAVE_DIR, model_name)
    if meta is None:
        continue
    m = meta["metrics"]
    records.append({
        "Model":      label,
        "Test Acc":   round(m.get("accuracy", 0), 4),
        "Test AUC":   round(m.get("auc", 0), 4),
        "Precision":  round(m.get("precision", 0), 4),
        "Recall":     round(m.get("recall", 0), 4),
        "F1":         round(m.get("f1", 0), 4),
        "Threshold":  round(meta.get("threshold", 0), 2),
        "Trainable Params": meta.get("model_params", {}).get("trainable", "—"),
    })

comparison_df = pd.DataFrame(records).set_index("Model")

metric_cols = ["Test Acc", "Test AUC", "Precision", "Recall", "F1"]
numeric_df  = comparison_df[metric_cols].apply(pd.to_numeric, errors="coerce")

print("=== Model Comparison ===")
print(comparison_df[["Threshold", "Trainable Params"]].to_string())

# Bar chart
ax = numeric_df[["Test AUC", "Recall", "Precision", "F1"]].plot(
    kind="bar", figsize=(12, 5), colormap="tab10", edgecolor="white"
)

colors_map = {"Recall": "tab:orange", "Precision": "tab:green", "F1": "tab:red"}

ax.set_ylim(0.5, 1.02)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Test Set Metrics")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

### Key Takeaways

- **Phase 1 (frozen base)** leverages ImageNet features as a fixed feature extractor. This provides a strong starting point even with limited medical data, since low-level features (edges, textures) transfer well across domains.
- **Phase 2 (fine-tuning block5)** adapts the highest-level convolutional features to the pneumonia domain, typically improving recall and F1 at the cost of slightly longer training.
- **Class weighting** (`{0: 1.0, 1: 4.0}`) and **threshold tuning** (PNEUMONIA F1) are critical to achieving high recall in this clinical setting, where missing a pneumonia case (false negative) has greater consequences than a false positive.